<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Recommender Systems Intuition</b>
</h1>
<div style="font-family:'Times New Roman';">
Netflix suggesting shows, amazon saying 'people also bought', spotify making playlists, these are all recommender systems. The whole job is to guess how much youd like something youve never seen, and then show you the stuff youd probably rate highest. Since i just did SVD this fits in nicely, becuase the fancy version of this is basicaly matrix factorization.
</div>

In [1]:
import numpy as np
import pandas as pd

## The user item matrix

Everything is built around one big table, users as rows, items (movies here) as columns, and the ratings inside. The catch is most of it is **empty**, becuase no one watches everything. The whole goal of a recommender is to fill in those blank cells with a good guess.

In [2]:
# a tiny ratings table, NaN means the user hasnt rated that movie
ratings = pd.DataFrame({
    'Matrix':    [5, 4, 1, np.nan],
    'Titanic':   [1, np.nan, 5, 4],
    'Inception': [5, 5, np.nan, 2],
    'Notebook':  [np.nan, 1, 5, 5],
}, index=['Ayush', 'Akshay', 'Shad', 'Aditya'])
ratings

,Matrix,Titanic,Inception,Notebook
Ayush,5.0,1.0,5.0,NaN
Akshay,4.0,NaN,5.0,1.0
Shad,1.0,5.0,NaN,5.0
Aditya,NaN,4.0,2.0,5.0


## Two main ways to do it

- **Content based** : look at the *features* of the items. If you liked a sci-fi action movie, recommend other sci-fi action movies. Needs you to actually have tags/features for everything.
- **Collaborative filtering** : ignore the item features completely and just use the crowd. If lots of people who rate things like you also loved some movie, youll probably love it too. This is the popular one and its what i'll build.

Collaborative filtering itself comes in two flavours:

- **user based** : find users similiar to you, recommend what they liked
- **item based** : find items similiar to ones you already liked

## Measuring similarity

Both flavours need a way to say how *similar* two users (or two items) are. A common one is **cosine similarity**, wich looks at the angle between two rating vectors. Pointing the same way means very similar (close to 1), at right angles means unrelated (0). Lets just compute it for two users by hand.

In [4]:
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# fill the blanks with 0 just for this quick similarity check
ayush = ratings.loc['Ayush'].fillna(0).values
akshay = ratings.loc['Akshay'].fillna(0).values
shad = ratings.loc['Shad'].fillna(0).values

print('Ayush vs Akshay:', round(cosine(ayush, akshay), 3))
print('Ayush vs Shad  :', round(cosine(ayush, shad), 3))

Ayush vs Akshay: 0.972
Ayush vs Shad  : 0.196


Ayush and Akshay come out way more similar than Ayush and Shad, wich makes sense, Ayush and Akshay both love the action/sci-fi stuff while Shad is more into romance. Thats the core trick, find similiar people and borrow their opinions.

Next i'll actually build user based collaborative filtering from scratch and use it to predict ratings and recommend movies.